# TrustDoc — Classifier fine-tuning on `rvl_cdip_mini`

Fine-tunes `microsoft/layoutlmv3-base` for document classification on `dvgodoy/rvl_cdip_mini` (1% subset of RVL-CDIP). Hyperparameters mirror `configs/classifier.yaml` in the repo.

**Different from the FUNSD throughput-verification notebook:** `rvl_cdip_mini` ships its own precomputed OCR (`ocr_words`/`word_boxes` columns), so this uses `apply_ocr=False` and feeds that OCR directly into the processor -- no Tesseract needed, and much faster than re-running OCR on 4,000 images. Word boxes are checked at runtime and normalized to LayoutLMv3's required 0-1000 scale only if they're not already in that range.

**Lessons applied here:** don't force a `transformers` version pin against Kaggle's own image; don't hardcode dataset column/split/label names or feature types (e.g. `label` here is a plain int column, not a `ClassLabel` -- names are derived from the `category` string column instead); only install packages that are actually missing and actually used.

**Decision thresholds:** if per-epoch time exceeds ~15 min, or val accuracy is unusable (<70%), stay on the mini subset. If training is fast and Kaggle quota allows, scale to the 10% subset as a stretch goal.

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True).stdout)
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU visible — on Kaggle: Settings > Accelerator > GPU T4x2/P100. On Colab: Runtime > Change runtime type > GPU.")

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
        print(f"{pkg}: already available, skipping install")
    except ImportError:
        print(f"{pkg}: not found, installing with --no-deps")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", pkg], check=True)

# datasets: load_dataset. accelerate: required by Trainer in recent transformers.
# (no tesseract/pytesseract needed -- rvl_cdip_mini ships precomputed OCR, see next cell)
ensure("datasets")
ensure("accelerate")

import transformers
print("transformers version (as provided by this environment):", transformers.__version__)


In [ ]:
from datasets import load_dataset

dataset = load_dataset("dvgodoy/rvl_cdip_mini")
print("splits:", list(dataset.keys()))

train_split = "train"
val_split = next((s for s in ("validation", "valid", "val") if s in dataset), None)
test_split = "test" if "test" in dataset else None
print("train/val/test splits detected:", train_split, val_split, test_split)

print("columns:", dataset[train_split].column_names)
print("features:", dataset[train_split].features)

image_column = next(c for c in ("image", "img", "pixel_values") if c in dataset[train_split].column_names)
label_column = next(c for c in ("label", "labels") if c in dataset[train_split].column_names)
print("using image column:", image_column, "| label column:", label_column)

# label column here is a plain int, not a ClassLabel (.names doesn't exist) -- but there's a
# "category" string column giving the human-readable name per example. Derive names from that
# instead of assuming a ClassLabel feature, and pool across all splits so no class is missed.
def label_category_pairs(split):
    return set(zip(dataset[split][label_column], dataset[split]["category"]))

pairs = label_category_pairs(train_split)
for s in (val_split, test_split):
    if s:
        pairs |= label_category_pairs(s)

label_to_name = dict(pairs)
num_labels = max(label_to_name) + 1
label_names = [label_to_name.get(i, f"unknown_{i}") for i in range(num_labels)]
print(f"num_labels: {num_labels}")
print("label names:", label_names)


In [ ]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForSequenceClassification

processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

# LayoutLMv3 requires boxes normalized to a 0-1000 scale (a documented common failure mode) --
# an out-of-range coordinate causes a CUDA "scatter gather kernel index out of bounds" crash.
# Checking a single sample isn't enough coverage across 3200+ examples with real-world OCR noise --
# scan the whole word_boxes column instead (cheap: it's Arrow column access, doesn't decode images).
_all_boxes = dataset[train_split]["word_boxes"]
_flat_coords = [c for boxes in _all_boxes for b in boxes for c in b]
_max_coord = max(_flat_coords) if _flat_coords else 0
_min_coord = min(_flat_coords) if _flat_coords else 0
print(f"train split word_box coordinate range: [{_min_coord}, {_max_coord}] across {len(_flat_coords)} values")
needs_normalization = _max_coord > 1000
print("word_boxes appear to be", "raw pixel coordinates -- will normalize to 0-1000" if needs_normalization
      else "already normalized to 0-1000 scale -- will use as-is")
if _max_coord > 1000 or _min_coord < 0:
    print("NOTE: out-of-range coordinates detected in the raw data -- clipping to [0, 1000] regardless "
          "of the normalization decision above, since a single bad box can crash the whole training run.")

def normalize_box(box, width, height):
    x0, y0, x1, y1 = box
    return [int(1000 * x0 / width), int(1000 * y0 / height), int(1000 * x1 / width), int(1000 * y1 / height)]

def clip_box(box):
    return [max(0, min(1000, c)) for c in box]

def prepare(examples):
    images = [img.convert("RGB") for img in examples[image_column]]  # LayoutLMv3 requires RGB, not grayscale
    words = examples["ocr_words"]
    if needs_normalization:
        boxes = [
            [normalize_box(b, w, h) for b in ex_boxes]
            for ex_boxes, w, h in zip(examples["word_boxes"], examples["width"], examples["height"])
        ]
    else:
        boxes = examples["word_boxes"]
    # always clip: cheap no-op for valid boxes, a real fix for any stray out-of-range outlier
    boxes = [[clip_box(b) for b in ex_boxes] for ex_boxes in boxes]
    encoding = processor(images, words, boxes=boxes, truncation=True, padding="max_length")
    encoding["labels"] = examples[label_column]
    return encoding

columns_to_remove = [c for c in dataset[train_split].column_names if c != label_column]

train_ds = dataset[train_split].map(prepare, batched=True, batch_size=8, remove_columns=columns_to_remove)
train_ds.set_format("torch")

if val_split:
    eval_ds = dataset[val_split].map(prepare, batched=True, batch_size=8, remove_columns=columns_to_remove)
    eval_ds.set_format("torch")
else:
    eval_ds = None
    print("WARNING: no validation split detected -- training without eval during training.")


In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer

model = LayoutLMv3ForSequenceClassification.from_pretrained(
    "microsoft/layoutlmv3-base", num_labels=num_labels
)

# Bake id2label/label2id in at construction time, from the same label_names computed above --
# so the first push_to_hub() call already carries real names (no separate "fix" pass later).
model.config.id2label = dict(enumerate(label_names))
model.config.label2id = {name: i for i, name in enumerate(label_names)}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

args = TrainingArguments(
    output_dir="results/classifier",
    fp16=True,  # T4 does not support bf16
    gradient_accumulation_steps=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    learning_rate=5e-5,
    seed=42,
    eval_strategy="epoch" if eval_ds is not None else "no",
    save_strategy="epoch" if eval_ds is not None else "no",
    # save_total_limit + load_best_model_at_end: without a limit, a full checkpoint (model +
    # optimizer state, ~1.5GB+) gets written every epoch with no cleanup, which exhausted
    # Kaggle's disk quota mid-run. Keeping only the 2 most recent/best also means the final
    # model is the best validation epoch, not just whatever epoch happened to run last.
    save_total_limit=2,
    load_best_model_at_end=eval_ds is not None,
    metric_for_best_model="accuracy" if eval_ds is not None else None,
    logging_steps=10,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics if eval_ds is not None else None,
)

import time
t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0
print(f"total train time: {elapsed/60:.1f} min ({elapsed/args.num_train_epochs/60:.2f} min/epoch)")
if elapsed / args.num_train_epochs > 15 * 60:
    print("WARNING: per-epoch time exceeds the 15 min scope threshold -- stay on the mini subset, do not scale to 10%.")

trainer.save_model("results/classifier")  # final (best) model only -- no optimizer state, much lighter

# Drop the per-epoch checkpoints (optimizer/scheduler state, ~1-1.5GB each) now that the final
# best model is saved above -- they're only useful for resuming training, not for anything
# downstream, and leaving several GB of output can break Kaggle's Output-tab file browser.
import glob, shutil
for ckpt_dir in glob.glob("results/classifier/checkpoint-*"):
    shutil.rmtree(ckpt_dir)
    print(f"removed {ckpt_dir}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

eval_source = eval_ds if eval_ds is not None else train_ds
predictions = trainer.predict(eval_source)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

acc = float((y_pred == y_true).mean())
print(f"accuracy on {'validation' if eval_ds is not None else 'train (no val split available)'}: {acc:.3f}")
if acc < 0.70:
    print("WARNING: accuracy is below the 70% scope threshold -- stay on the mini subset and investigate before scaling.")

# Per-class precision/recall/F1, saved in the same shape as the extractor's
# results/extractor_report.json -- keeps the two models' result artifacts symmetric
# instead of the classifier only getting a confusion-matrix image.
import json

report = classification_report(
    y_true, y_pred, target_names=label_names, output_dict=True, zero_division=0
)
with open("results/classifier_report.json", "w") as f:
    json.dump(report, f, indent=2)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, xticks_rotation="vertical", colorbar=False)
plt.tight_layout()
plt.savefig("results/classifier_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
from huggingface_hub import login, HfApi

HF_REPO_ID = "vxa8502/trustdoc-classifier"

# Prefer a Kaggle Secret (notebook editor: Add-ons > Secrets, key "HF_TOKEN") so the token
# never ends up in plaintext in the notebook. Falls back to an interactive prompt if unset.
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    from getpass import getpass
    hf_token = getpass("No HF_TOKEN Kaggle Secret found -- paste a Hugging Face WRITE token: ")

login(token=hf_token)

trainer.model.push_to_hub(HF_REPO_ID, private=False)
processor.push_to_hub(HF_REPO_ID, private=False)

# LayoutLMv3-base is CC BY-NC-SA 4.0 (non-commercial, share-alike) -- a fine-tuned derivative
# inherits that license, so the model card must say so explicitly rather than defaulting to none.
model_card = f"""---
license: cc-by-nc-sa-4.0
base_model: microsoft/layoutlmv3-base
datasets:
- dvgodoy/rvl_cdip_mini
metrics:
- accuracy
---

# TrustDoc Document Classifier

LayoutLMv3-base fine-tuned for document type classification (16 RVL-CDIP classes), part of the
TrustDoc project -- a document AI trust layer with calibrated confidence for human-in-the-loop review.

- **Base model:** [microsoft/layoutlmv3-base](https://huggingface.co/microsoft/layoutlmv3-base)
  (CC BY-NC-SA 4.0 -- **non-commercial use only**, inherited by this fine-tune)
- **Training data:** [dvgodoy/rvl_cdip_mini](https://huggingface.co/datasets/dvgodoy/rvl_cdip_mini),
  a 1% subset of RVL-CDIP (16 document classes)
- **Validation accuracy:** 85.8%

## Limitations

RVL-CDIP has documented issues that affect this model: shortcut-feature bias (some predictions may
rely on spurious per-page ID codes rather than content), ~8% label noise, train/test duplication,
real PII in the source documents, and a tobacco-industry/1950s-2002-only domain -- generalization
to modern documents is unproven.
"""

HfApi().upload_file(
    path_or_fileobj=model_card.encode(),
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="model",
)
print(f"pushed to https://huggingface.co/{HF_REPO_ID}")


## Checklist
- [ ] Detected columns/splits/label count printed above and look right (no blind assumptions)
- [ ] word_box coordinate range printed above — confirm whether clipping/normalization actually kicked in
- [ ] Per-epoch time recorded — under 15 min means the mini subset is fine
- [ ] Validation accuracy recorded (best epoch, via `load_best_model_at_end`) — at or above 70% means proceed; below means investigate before scaling to the 10% subset
- [ ] Training completed without a disk-full checkpoint-write crash (bounded by `save_total_limit=2`)
- [ ] Confusion matrix saved to `results/classifier_confusion_matrix.png`
- [ ] Per-class report saved to `results/classifier_report.json` (same shape as the extractor's `results/extractor_report.json`)
- [ ] Record accuracy, per-epoch time, and the scale-up decision
- [ ] Model pushed to https://huggingface.co/vxa8502/trustdoc-classifier and loads back correctly
- [ ] Download the trained checkpoint from `results/classifier` before the Kaggle session ends (session storage is not persistent across sessions unless saved as a Kaggle Dataset/output)
